In [8]:
# 1 Load the spreadsheet

import pandas as pd

df = pd.read_excel("raw_data.xlsx", sheet_name=0) # or: pd.read_csv("raw_data.csv")
print(df.shape)
print(df.head())

# 2 Standardize column names (huge quality of life win)

df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(r"\s+", "-", regex=True)
      .str.replace(r"[⌃a-z0-9_]", "", regex=True)
)

# 3 Trim whitespace and normalize text

text_cols = df.select_dtypes(include="object").columns
df[text_cols] = df[text_cols].apply(lambda s: s.str.strip())


# Common normalizations

df["state"] = df["state"].str.upper()
df["email"] = df["email"].str.lower()

# 4 Fix data types (numbers, dates, money, #'s that are dirty with commas, dollar signs, parentheses)

df["amount"] = (
    df["amount"].astype(str)
      .str.replace(r"[\$,]", "", regex=True)
      .str.replace(r"\((.*)", r"-\1", regex=True) # (123) -> -123
)

df["amount"] = pd.to_numeric(df["amount"], errors="coerce")

### DATES ###

df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")

#5 Handle missing values
   # Quick audit

missig = df.isna().sum().sort_values(ascending=False)
print(missing.head(20))

  # Strategies
df = df.drop.na(subset=["email"])  # must have column
df["amount"] = df["amount"].fillna(0) # numeric default
df["state"]  = df["state"].fillna("UNK") # categorical default

   #forward-fill (good for repeated labels):

df["account_id"] = df["account_id"].ffill()

#6 Remove duplicates

  # exact duplicate rows

df = df.drop_duplicates()

  # duplicates based on certain columns
df = df.drop_duplicates(subset=["email"], keep="last")

#7 Validate / flag bad rows
  # exaple checks
bad_amount = df["amount"].lt(0)    # negative amounts
bad_email = ~df["email"].str.contains(r"@", na=False)

df["issue"] = ""
df.loc[bad_amount, "issue"] += "negative_amount;"
df.loc[bad_email,  "issue"] += "bad_email;"

# Keep a clean set and a “needs review” set:

review = df[df["issue"] != ""].copy()
clean  = df[df["issue"] == ""].copy()
# 8) Format / sort / select final columns
clean = clean.sort_values(["order_date", "amount"], ascending=[True, False])

final_cols = ["order_date", "customer", "email", "state", "amount"]
clean = clean[final_cols]
# 9) Export back to Excel (with real Excel formatting)

# Pandas alone writes data, but xlsxwriter lets you format nicely:

with pd.ExcelWriter("cleaned.xlsx", engine="xlsxwriter") as writer:
    clean.to_excel(writer, index=False, sheet_name="Clean")
    review.to_excel(writer, index=False, sheet_name="Review")

    wb = writer.book
    ws = writer.sheets["Clean"]

    money_fmt = wb.add_format({"num_format": "$#,##0.00"})
    date_fmt  = wb.add_format({"num_format": "yyyy-mm-dd"})
    header_fmt = wb.add_format({"bold": True, "bg_color": "#D9E1F2"})

    # header format
    for col, name in enumerate(clean.columns):
        ws.write(0, col, name, header_fmt)

    # set column formats (adjust letters/indices to your columns)
    ws.set_column("A:A", 12, date_fmt)     # order_date
    ws.set_column("E:E", 12, money_fmt)    # amount
    ws.autofilter(0, 0, len(clean), len(clean.columns) - 1)
    ws.freeze_panes(1, 0)
# A reusable “cleaning template” function
import pandas as pd

def clean_sheet(path_in: str, path_out: str):
    df = pd.read_excel(path_in)

    # column names
    df.columns = (df.columns.str.strip().str.lower()
                  .str.replace(r"\s+", "_", regex=True)
                  .str.replace(r"[^a-z0-9_]", "", regex=True))

    # text cleanup
    text_cols = df.select_dtypes(include="object").columns
    df[text_cols] = df[text_cols].apply(lambda s: s.str.strip())

    # example: types
    if "order_date" in df.columns:
        df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")

    if "amount" in df.columns:
        df["amount"] = (df["amount"].astype(str)
                        .str.replace(r"[\$,]", "", regex=True)
                        .str.replace(r"\((.*)\)", r"-\1", regex=True))
        df["amount"] = pd.to_numeric(df["amount"], errors="coerce").fillna(0)

    # duplicates
    df = df.drop_duplicates()

    df.to_excel(path_out, index=False)
    return df

cleaned = clean_sheet("raw_data.xlsx", "cleaned.xlsx")
print(cleaned.head())
















FileNotFoundError: [Errno 2] No such file or directory: 'raw_data.xlsx'